In [17]:
import json

In [38]:
import json
import random
import re
from collections import defaultdict
from pathlib import Path
from sklearn.model_selection import train_test_split

# -------------------------
# CONFIG
# -------------------------
INPUT_JSON = "/home/simonettos/thijs/classification/classification/datasets/bosch_train.json"
VAL_OUT    = "/home/simonettos/thijs/classification/classification/datasets/bosch_split_val_hmcat.json"

# technique_id -> description json
DESCRIPTIONS_JSON = "/home/simonettos/thijs/data_augmentatio_stefano/mitre/ttp_descriptions2.json"

# batch output (one JSON object per line)
BATCH_JSONL_OUT = "/home/simonettos/thijs/classification/classification/datasets/bosch_hmcat_chatgpt_batch.jsonl"

SEED = 0
VAL_FRAC = 0.20

# Underrepresented definition
M_MIN_SAMPLES = 49          # labels with < M are considered scarce
LABEL_PREFIX  = None         # e.g. "T1" to restrict, or None for all techniques

# Prompt construction
N_DEMOS = 3                  # number of demo examples in prompt
N_NEW_PER_LABEL = 10         # ask model for this many new sentences per label
MAX_LABELS = None            # cap number of labels to include (None = all scarce labels)

# Prompt / description controls
MAX_DESC_CHARS = 900         # trim long MITRE technique descriptions to keep prompts small
STRIP_CITATIONS = True       # remove "(Citation: ...)" chunks and markdown links for cleanliness

N_NEW_PER_SENTENCE = 9   # <-- 9 per underrepresented sentence
MAX_REQUESTS = None      # optionally cap total requests (e.g., 5000)

# OpenAI request config (Batch API style)
OPENAI_MODEL = "gpt-5-mini"  # change as you like
OPENAI_URL = "/v1/responses"   # Responses API endpoint

random.seed(SEED)

# -------------------------
# DATA IO (columnar JSON <-> row list)
# -------------------------
def load_columnar_json(path: str) -> list[dict]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    sent = data["sentence"]
    labs = data["labels"]
    docs = data.get("doc_title", {})

    keys = sorted(sent.keys(), key=lambda x: int(x))
    rows = []
    for k in keys:
        rows.append({
            "sentence": sent[k],
            "labels": labs.get(k, []),
            "doc_title": docs.get(k, "")
        })
    return rows

def save_columnar_json(rows: list[dict], path: str) -> None:
    out = {"sentence": {}, "labels": {}, "doc_title": {}}
    for i, r in enumerate(rows):
        k = str(i)
        out["sentence"][k] = r["sentence"]
        out["labels"][k] = r["labels"]
        out["doc_title"][k] = r.get("doc_title", "")
    with open(path, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)

# -------------------------
# HELPERS
# -------------------------
def iter_labels(row: dict) -> list[str]:
    labs = row.get("labels", [])
    if not isinstance(labs, list):
        return []
    out = []
    for l in labs:
        if isinstance(l, str):
            l = l.strip()
            if l:
                out.append(l)
    return out

def is_technique_label(lab: str) -> bool:
    # Keeps Txxxx and Txxxx.xxx; ignores TAxxxx etc
    return lab.startswith("T") and len(lab) >= 5 and lab[1:5].isdigit()

def label_allowed(lab: str) -> bool:
    if not is_technique_label(lab):
        return False
    if LABEL_PREFIX is None:
        return True
    return lab.startswith(LABEL_PREFIX)

def load_descriptions(path: str) -> dict[str, str]:
    with open(path, "r", encoding="utf-8") as f:
        d = json.load(f)
    # normalize keys & ensure strings
    out = {}
    for k, v in d.items():
        if isinstance(k, str) and isinstance(v, str):
            out[k.strip()] = v.strip()
    return out

def clean_description(desc: str) -> str:
    s = desc

    if STRIP_CITATIONS:
        # remove "(Citation: ...)" occurrences (common in ATT&CK text)
        s = re.sub(r"\(Citation:[^)]+\)", "", s)

    # remove markdown links but keep anchor text: [text](url) -> text
    s = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", s)

    # remove html tags like <code>...</code>
    s = re.sub(r"<[^>]+>", "", s)

    # collapse whitespace
    s = re.sub(r"\s+", " ", s).strip()

    if len(s) > MAX_DESC_CHARS:
        s = s[:MAX_DESC_CHARS].rstrip() + "…"
    return s

INSTRUCTION = (
    "You are generating synthetic Cyber Threat Intelligence (CTI) sentences.\n"
    "Goal: produce realistic, concise CTI-style sentences that would map to the SAME MITRE ATT&CK technique.\n"
    "Rules:\n"
    "- Do NOT copy any demo sentence verbatim.\n"
    "- Keep it plausible and technical (malware behavior, commands, persistence, credential access, etc.).\n"
    "- Do NOT mention 'MITRE' or 'ATT&CK' explicitly.\n"
    "- Output ONLY a JSON array of strings, no extra text.\n"
)

from typing import Optional, List

def build_prompt(label: str, label_desc: Optional[str], demos: List[str], n_new: int) -> str:
    parts = [INSTRUCTION, f"Technique ID: {label}"]
    if label_desc:
        parts.append(f"Technique description (for context): {label_desc}")
    parts.append("\nInspiration sentence (do not copy):")
    for i, d in enumerate(demos, 1):
        parts.append(f"{i}. {d.strip()}")
    parts.append(f"\nTask: Generate {n_new} NEW CTI sentences for Technique {label}.")
    parts.append('Output format: ["sentence1", "sentence2", ...]')
    return "\n".join(parts)


# -------------------------
# SPLIT TRAIN/VAL (row-level)
# -------------------------
rows = load_columnar_json(INPUT_JSON)
train_rows, val_rows = train_test_split(rows, test_size=VAL_FRAC, random_state=SEED)
save_columnar_json(val_rows, VAL_OUT)
print(f"Split complete: train={len(train_rows)} val={len(val_rows)}")

# -------------------------
# LOAD DESCRIPTIONS
# -------------------------
desc_map = load_descriptions(DESCRIPTIONS_JSON)

# -------------------------
# LABEL COUNTS ON TRAIN
# -------------------------
label2rows = defaultdict(list)
for r in train_rows:
    for lab in iter_labels(r):
        if label_allowed(lab):
            label2rows[lab].append(r)

label_counts = {lab: len(rs) for lab, rs in label2rows.items()}
scarce = [lab for lab, c in label_counts.items() if c < M_MIN_SAMPLES]
scarce.sort(key=lambda x: label_counts[x])  # scarcest first

if MAX_LABELS is not None:
    scarce = scarce[:MAX_LABELS]

print(f"Eligible technique labels in train: {len(label_counts)}")
print(f"Scarce labels (<{M_MIN_SAMPLES}): {len(scarce)}")

import hashlib

# -------------------------
# WRITE OPENAI BATCH JSONL (9 per UNDERREPRESENTED SENTENCE)
# -------------------------
out_path = Path(BATCH_JSONL_OUT)
out_path.parent.mkdir(parents=True, exist_ok=True)

scarce_set = set(scarce)

def short_hash(s: str) -> str:
    return hashlib.sha1(s.encode("utf-8")).hexdigest()[:10]

n_written = 0
n_skipped_empty = 0
n_missing_desc = 0

with out_path.open("w", encoding="utf-8") as f:
    # iterate over ALL train rows; create one request per (row, scarce label)
    for idx, r in enumerate(train_rows):
        sent = r.get("sentence", "")
        if not isinstance(sent, str) or not sent.strip():
            n_skipped_empty += 1
            continue

        labs = [lab for lab in iter_labels(r) if lab in scarce_set]
        if not labs:
            continue

        # If a sentence has multiple scarce labels, you likely want separate requests
        # so you can assign outputs to each label cleanly.
        for lab in labs:
            pool = [
                rr["sentence"] for rr in label2rows[lab]
                if isinstance(rr.get("sentence"), str) and rr["sentence"].strip()
            ]
            if not pool:
                continue

            # demos: include the source sentence + a couple extra examples for that label (excluding source)
            others = [p for p in pool if p.strip() != sent.strip()]
            extra = others if len(others) <= (N_DEMOS - 1) else random.sample(others, N_DEMOS - 1)
            demos = [sent.strip()] + extra

            raw_desc = desc_map.get(lab)
            if raw_desc is None:
                n_missing_desc += 1
                cleaned_desc = None
            else:
                cleaned_desc = clean_description(raw_desc)

            prompt = build_prompt(lab, cleaned_desc, demos, N_NEW_PER_SENTENCE)

            req = {
                "custom_id": f"augment__{lab}__row{idx}__{short_hash(sent)}",
                "method": "POST",
                "url": OPENAI_URL,
                "body": {
                    "model": OPENAI_MODEL,
                    "input": prompt
                }
            }

            f.write(json.dumps(req, ensure_ascii=False) + "\n")
            n_written += 1

            if MAX_REQUESTS is not None and n_written >= MAX_REQUESTS:
                break

        if MAX_REQUESTS is not None and n_written >= MAX_REQUESTS:
            break

print(f"Wrote {n_written} requests (per underrepresented sentence) to JSONL: {BATCH_JSONL_OUT}")
if n_skipped_empty:
    print(f"Skipped empty sentences: {n_skipped_empty}")
if n_missing_desc:
    print(f"Warning: {n_missing_desc} requests had missing technique descriptions in {DESCRIPTIONS_JSON}")


Split complete: train=2934 val=734
Eligible technique labels in train: 111
Scarce labels (<49): 104
Wrote 747 requests (per underrepresented sentence) to JSONL: /home/simonettos/thijs/classification/classification/datasets/bosch_hmcat_chatgpt_batch.jsonl


In [39]:
#!/usr/bin/env python3
"""
Check and enforce uniqueness of `custom_id` in an OpenAI Batch JSONL file.

Modes:
- check_only=True:   just report duplicates and exit non-zero if any
- check_only=False:  rewrite file with de-duplicated custom_ids by appending a stable suffix

Notes:
- Keeps every request line (does NOT drop duplicates). It only changes custom_id for collisions.
- Suffix is deterministic per line content, so reruns are stable.
"""

from __future__ import annotations

import json
import hashlib
from pathlib import Path
from typing import Dict, List, Tuple


# -------------------------
# CONFIG
# -------------------------
IN_JSONL  = "/home/simonettos/thijs/classification/classification/datasets/bosch_hmcat_chatgpt_batch.jsonl"
OUT_JSONL = "/home/simonettos/thijs/classification/classification/datasets/bosch_hmcat_chatgpt_batch.jsonl"

CHECK_ONLY = False  # set True to only report duplicates, no rewrite


def stable_line_fingerprint(obj: dict) -> str:
    """
    Stable fingerprint for a request object.
    We exclude custom_id itself so collisions can be resolved deterministically.
    """
    tmp = dict(obj)
    tmp.pop("custom_id", None)
    canon = json.dumps(tmp, ensure_ascii=False, sort_keys=True, separators=(",", ":"))
    return hashlib.sha1(canon.encode("utf-8")).hexdigest()[:10]


def load_jsonl(path: str) -> List[dict]:
    items: List[dict] = []
    with open(path, "r", encoding="utf-8") as f:
        for ln, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue
            try:
                items.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise SystemExit(f"[ERROR] Invalid JSON on line {ln}: {e}") from e
    return items


def find_duplicates(items: List[dict]) -> Dict[str, List[int]]:
    """
    Returns: custom_id -> list of indices (0-based) where it appears, only for duplicates.
    """
    seen: Dict[str, List[int]] = {}
    for i, obj in enumerate(items):
        cid = obj.get("custom_id")
        if not isinstance(cid, str) or not cid.strip():
            # treat missing/empty custom_id as a collision bucket too
            cid = ""
        seen.setdefault(cid, []).append(i)
    return {cid: idxs for cid, idxs in seen.items() if len(idxs) > 1}


def enforce_unique_custom_ids(items: List[dict]) -> Tuple[List[dict], int]:
    """
    For any duplicate custom_id, keep the first occurrence unchanged.
    For subsequent occurrences, append a deterministic suffix:
      <orig>__dup<k>__<fingerprint>
    Returns modified items and number of changed lines.
    """
    used = set()
    changed = 0

    for obj in items:
        cid = obj.get("custom_id")
        if not isinstance(cid, str):
            cid = ""

        base = cid if cid.strip() else "missing_custom_id"
        if base not in used:
            obj["custom_id"] = base
            used.add(base)
            continue

        # collision: create a unique id
        fp = stable_line_fingerprint(obj)
        k = 2
        new_cid = f"{base}__dup{k}__{fp}"
        while new_cid in used:
            k += 1
            new_cid = f"{base}__dup{k}__{fp}"

        obj["custom_id"] = new_cid
        used.add(new_cid)
        changed += 1

    return items, changed


def write_jsonl(items: List[dict], path: str) -> None:
    out_path = Path(path)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    with out_path.open("w", encoding="utf-8") as f:
        for obj in items:
            f.write(json.dumps(obj, ensure_ascii=False) + "\n")


def main() -> None:
    items = load_jsonl(IN_JSONL)
    dups = find_duplicates(items)

    if not dups:
        print(f"[OK] No duplicate custom_id found in: {IN_JSONL} (lines={len(items)})")
        return

    # Report
    total_dup_lines = sum(len(v) for v in dups.values())
    print(f"[WARN] Found {len(dups)} duplicated custom_id values affecting {total_dup_lines} lines.")
    # Print a small sample
    for cid, idxs in list(dups.items())[:20]:
        print(f"  - custom_id={repr(cid)} occurs {len(idxs)} times at indices: {idxs[:10]}{'...' if len(idxs)>10 else ''}")
    if len(dups) > 20:
        print(f"  (showing first 20 duplicate groups)")

    if CHECK_ONLY:
        raise SystemExit(2)

    # Enforce + rewrite
    items2, changed = enforce_unique_custom_ids(items)
    write_jsonl(items2, OUT_JSONL)

    # Re-check
    dups2 = find_duplicates(items2)
    if dups2:
        raise SystemExit(f"[ERROR] Still found duplicates after rewrite (unexpected). Example: {next(iter(dups2.items()))}")

    print(f"[FIXED] Rewrote JSONL with unique custom_id:")
    print(f"  input : {IN_JSONL}")
    print(f"  output: {OUT_JSONL}")
    print(f"  changed custom_id on {changed} lines.")


if __name__ == "__main__":
    main()


[WARN] Found 26 duplicated custom_id values affecting 52 lines.
  - custom_id='augment__T1496__row451__059f4d269e' occurs 2 times at indices: [97, 98]
  - custom_id='augment__T1087__row687__a1c3d1dfd0' occurs 2 times at indices: [153, 154]
  - custom_id='augment__T1555__row936__4931c9115b' occurs 2 times at indices: [203, 204]
  - custom_id='augment__T1137__row1225__4bc5b5bc2f' occurs 2 times at indices: [268, 269]
  - custom_id='augment__T1218__row1284__9ded1f7224' occurs 2 times at indices: [288, 289]
  - custom_id='augment__T1568__row1541__488ba1fb78' occurs 2 times at indices: [354, 355]
  - custom_id='augment__T1204__row1624__ac4fde1246' occurs 2 times at indices: [387, 388]
  - custom_id='augment__T1497__row1652__a65767d4da' occurs 2 times at indices: [392, 393]
  - custom_id='augment__T1573__row1902__cb5fae7137' occurs 2 times at indices: [448, 449]
  - custom_id='augment__T1090__row1957__c01f63d0a1' occurs 2 times at indices: [459, 460]
  - custom_id='augment__T1083__row2003__9

Parse the output

In [14]:
#!/usr/bin/env python3
from __future__ import annotations

import json
import re
import hashlib
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Optional, Tuple


# -------------------------
# CONFIG
# -------------------------
INPUT_JSON = "/home/simonettos/thijs/classification/classification/datasets/tram_train.json"
BATCH_OUTPUT_JSONL = "/home/simonettos/thijs/classification/classification/batch_699435ee26bc8190b75bb41e670aa44d_output.jsonl"
OUT_JSON = "/home/simonettos/thijs/classification/classification/datasets/tram_augmented_hcmat_artificial.json"

GROUP_BY = "source_sentence"   # new option



# -------------------------
# DATA IO
# -------------------------
def load_columnar_json(path: str) -> List[dict]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)

    sent = data["sentence"]
    labs = data["labels"]
    docs = data.get("document", {})

    keys = sorted(sent.keys(), key=lambda x: int(x))
    rows = []
    for k in keys:
        rows.append(
            {
                "orig_id": k,  # keep original id
                "sentence": sent[k],
                "labels": labs.get(k, []),
                "document": docs.get(k, ""),
            }
        )
    return rows


def short_hash(s: str) -> str:
    return hashlib.sha1(s.encode("utf-8")).hexdigest()[:10]


def build_hash_index(rows: List[dict]) -> Dict[str, List[dict]]:
    """
    Map sha1[:10] -> list of rows (collisions are possible, so store list).
    """
    idx: Dict[str, List[dict]] = defaultdict(list)
    for r in rows:
        s = r.get("sentence", "")
        if isinstance(s, str) and s.strip():
            idx[short_hash(s.strip())].append(r)
    return idx


# -------------------------
# Batch helpers
# -------------------------
CID_RE = re.compile(r"^augment__([^_]+)__row(\d+)__([0-9a-f]{10})$")

def parse_custom_id(custom_id: str) -> Optional[Tuple[str, int, str]]:
    """
    custom_id: augment__{lab}__row{idx}__{sha1_10}
    Returns (lab, idx, sha10)
    """
    if not isinstance(custom_id, str):
        return None
    m = CID_RE.match(custom_id)
    if not m:
        return None
    return m.group(1), int(m.group(2)), m.group(3)


def extract_output_text(resp_body: dict) -> Optional[str]:
    try:
        out = resp_body.get("output", [])
        for item in out:
            if item.get("type") == "message":
                for c in item.get("content", []):
                    if c.get("type") == "output_text":
                        return c.get("text")
    except Exception:
        return None
    return None


def parse_json_array_of_strings(text: str) -> Optional[List[str]]:
    if not isinstance(text, str):
        return None
    s = text.strip()
    try:
        arr = json.loads(s)
    except json.JSONDecodeError:
        return None
    if not isinstance(arr, list):
        return None
    out: List[str] = []
    for x in arr:
        if isinstance(x, str) and x.strip():
            out.append(x.strip())
    return out


# -------------------------
# Main
# -------------------------
def main() -> None:
    rows = load_columnar_json(INPUT_JSON)
    hash_index = build_hash_index(rows)

    grouped: Dict[str, List[dict]] = defaultdict(list)

    n_lines = 0
    n_ok = 0
    n_skip_non200 = 0
    n_skip_bad_cid = 0
    n_skip_no_text = 0
    n_skip_bad_json = 0
    n_skip_no_match = 0
    n_hash_collisions = 0

    with open(BATCH_OUTPUT_JSONL, "r", encoding="utf-8") as f:
        for line in f:
            n_lines += 1
            line = line.strip()
            if not line:
                continue

            try:
                obj = json.loads(line)
            except json.JSONDecodeError:
                continue

            resp = obj.get("response") or {}
            if resp.get("status_code") != 200:
                n_skip_non200 += 1
                continue

            parsed = parse_custom_id(obj.get("custom_id"))
            if not parsed:
                n_skip_bad_cid += 1
                continue
            lab, _row_idx_unused, sha10 = parsed

            candidates = hash_index.get(sha10, [])
            if not candidates:
                n_skip_no_match += 1
                continue
            if len(candidates) > 1:
                # rare, but possible sha1[:10] collision or duplicated sentences
                n_hash_collisions += 1
            row = candidates[0]  # pick first; if you want, we can disambiguate further

            body = resp.get("body") or {}
            text = extract_output_text(body)
            if not text:
                n_skip_no_text += 1
                continue

            new_sents = parse_json_array_of_strings(text)
            if new_sents is None:
                n_skip_bad_json += 1
                continue

            document = (row.get("document") or "").strip()

            # IMPORTANT: use the raw source sentence as the dict key
            # If you want to preserve exact punctuation/spacing, do NOT normalize here.
            source_sentence = (row.get("sentence") or "").strip()

            if GROUP_BY == "source_sentence":
                key = source_sentence
            elif GROUP_BY == "document":
                key = document if document else source_sentence
            elif GROUP_BY == "sentence":
                # (legacy) same as source_sentence really
                key = source_sentence
            else:
                raise ValueError("GROUP_BY must be 'source_sentence', 'document', or 'sentence'")

            for s in new_sents:
                grouped[key].append(
                    {
                        "augmented_sentence": s,
                        "labels": [lab],          # label you requested in this augmentation job
                        "source_sentence": source_sentence,  # optional: helps debugging/traceability
                        "source_labels": row.get("labels", []),  # optional: original labels
                    }
                )

            n_ok += 1

    out_path = Path(OUT_JSON)
    out_path.parent.mkdir(parents=True, exist_ok=True)
    out_path.write_text(json.dumps(grouped, ensure_ascii=False, indent=2), encoding="utf-8")

    print(f"Read lines: {n_lines}")
    print(f"OK: {n_ok}")
    print(f"Skipped non-200: {n_skip_non200}")
    print(f"Skipped bad custom_id: {n_skip_bad_cid}")
    print(f"Skipped missing output_text: {n_skip_no_text}")
    print(f"Skipped invalid JSON array: {n_skip_bad_json}")
    print(f"Skipped no hash match in INPUT_JSON: {n_skip_no_match}")
    print(f"Hash collisions/duplicates observed: {n_hash_collisions}")
    print(f"Wrote: {OUT_JSON}")
    print(f"Groups: {len(grouped)}")


if __name__ == "__main__":
    main()


Read lines: 2364
OK: 1233
Skipped non-200: 0
Skipped bad custom_id: 0
Skipped missing output_text: 0
Skipped invalid JSON array: 7
Skipped no hash match in INPUT_JSON: 1124
Hash collisions/duplicates observed: 2
Wrote: /home/simonettos/thijs/classification/classification/datasets/tram_augmented_hcmat_artificial.json
Groups: 1015


In [23]:
import os
os.environ["HTTP_PROXY"]= "http://proxy.utwente.nl:3128"
os.environ["HTTPS_PROXY"]= "http://proxy.utwente.nl:3128"
os.environ["http_proxy"]= "http://proxy.utwente.nl:3128"
os.environ["https_proxy"]= "http://proxy.utwente.nl:3128"

In [32]:
import json

with open("/home/simonettos/thijs/classification/classification/datasets/bosch_augmented_hcmat_artificial.json", "r") as f:
    artificial_data = json.load(f)

len(artificial_data)

628

In [33]:
import pandas as pd
from sklearn.model_selection import train_test_split

test_size = 0.2
random_state = 0

tram_df = pd.read_json("datasets/bosch_train.json")
df_train, df_val = train_test_split(tram_df, test_size=test_size, random_state=random_state)
df_train.shape, df_val.shape

((2934, 4), (734, 4))

In [17]:
import loader

model = loader.load_model_for_embedding("sentence-transformers/all-mpnet-base-v2")

Models ['bert-base-uncased', 'bert-base-cased', 'FacebookAI/roberta-base', 'FacebookAI/xlm-roberta-base', 'FacebookAI/roberta-large', 'FacebookAI/xlm-roberta-large', 's2w-ai/DarkBERT', 'jackaduma/SecBERT', 'jackaduma/SecRoBERTa', 'markusbayer/CySecBERT', 'allenai/scibert_scivocab_cased', 'allenai/scibert_scivocab_uncased', 'priyankaranade/cybert', 'tram_multi_label_model', 'ehsanaghaei/SecureBERT'] MODEL_SENTENCE_SIM ['sentence-transformers/all-mpnet-base-v2', 'basel/ATTACK-BERT', 'qcri-cs/SentSecBert_10k']
Loading model: sentence-transformers/all-mpnet-base-v2 ...


In [34]:
from collections import Counter

# Flatten the list of labels and count the occurrences of each label
label_counts = Counter(label for labels in tram_df['labels'] for label in labels)
print(len(tram_df['labels']))
# Convert the counter to a DataFrame for better visualization
label_distribution = pd.DataFrame.from_dict(label_counts, orient='index', columns=['count']).sort_values(by='count', ascending=False)
print(label_distribution)

3668
        count
TA0011    407
T1566     273
T1059     206
T1486     168
T1105     117
...       ...
G0083       1
S0561       1
G0040       1
G0062       1
G0026       1

[203 rows x 1 columns]


In [37]:
import numpy as np
from tqdm import tqdm
from sentence_transformers import util

alpha = 0.3
beta = 0.8

artificial_data_selected = []

for _, row in tqdm(df_train.iterrows(), total=df_train.shape[0]):
    sentence = row["sentence"]
    labels = row["labels"]

    artificial_sent_w_labels = artificial_data.get(sentence, [])
    if not artificial_sent_w_labels:   # <-- important
        continue

    augmented_sentences = [item["augmented_sentence"] for item in artificial_sent_w_labels]
    # if they exist but are empty/blank strings, filter
    augmented_sentences = [s for s in augmented_sentences if isinstance(s, str) and s.strip()]
    if not augmented_sentences:        # <-- also important
        continue

    sent_emb = model.encode(sentence, convert_to_tensor=True)
    aug_embs = model.encode(augmented_sentences, convert_to_tensor=True)

    sim = util.cos_sim(sent_emb, aug_embs).cpu().numpy().reshape(-1)  # shape: (n_aug,)

    indices = np.where((sim >= alpha) & (sim <= beta))[0]
    if len(indices) < 1:
        continue

    if not labels:  # no labels -> pick 1 random
        rand_i = np.random.choice(indices)
        artificial_data_selected.append(artificial_sent_w_labels[rand_i])
    else:
        for i in indices:
            artificial_data_selected.append(artificial_sent_w_labels[i])

from collections import Counter

# Get the index with the maximum similarity
# Extract labels from artificial_data_selected
selected_labels = [label for item in artificial_data_selected for label in item['labels']]

# Count the occurrences of each label
selected_label_counts = Counter(selected_labels)

# Convert the counter to a DataFrame for better visualization
selected_label_distribution = pd.DataFrame.from_dict(selected_label_counts, orient='index', columns=['count']).sort_values(by='count', ascending=False)
print(selected_label_distribution)
import random
from collections import Counter

artificial = {"sentence": [], "labels": []}
shortages = {}

for label in label_distribution.index:
    target = int(label_distribution.loc[label, "count"])

    selected = [item for item in artificial_data_selected if label in item.get("labels", [])]
    avail = len(selected)

    if avail == 0:
        shortages[label] = (target, 0)
        continue

    k = min(target, avail)
    if k < target:
        shortages[label] = (target, avail)

    sampled = random.sample(selected, k)

    artificial["sentence"].extend([x["augmented_sentence"] for x in sampled])
    artificial["labels"].extend([x["labels"] for x in sampled])

# add the no-label augmentations
for x in artificial_data_selected:
    if not x.get("labels"):
        artificial["sentence"].append(x["augmented_sentence"])
        artificial["labels"].append(x["labels"])

print(f"Labels with shortages: {len(shortages)}")
if shortages:
    # show a few
    for lab, (t, a) in list(shortages.items())[:100]:
        print(f"{lab}: wanted {t}, had {a}")


for data in artificial_data_selected:
    if len(data['labels']) == 0:
        artificial['sentence'].append(data['augmented_sentence'])
        artificial['labels'].append(data['labels'])
artificial_df = pd.DataFrame(artificial)
artificial_df.drop_duplicates(subset=['sentence'], inplace=True)
artificial_df

augmented_df = pd.concat([df_train, artificial_df], ignore_index=True)
augmented_df.reset_index(drop=True, inplace=True)
augmented_df
augmented_df['document'].fillna('artificial_data', inplace=True)
# Shuffle the dataframe
augmented_df_shuffled = augmented_df.sample(frac=1, random_state=random_state).reset_index(drop=True)

# Save to JSON file
augmented_df_shuffled.to_json("datasets/bosch_train_augmented_artificial.json")

  0%|          | 0/2934 [00:00<?, ?it/s]

100%|██████████| 2934/2934 [00:32<00:00, 90.33it/s] 


       count
T1190    211
T1573    200
T1204    191
T1056    168
T1041    154
...      ...
T1087      4
T1437      4
T1563      4
T1119      3
T1120      3

[103 rows x 1 columns]
Labels with shortages: 100
TA0011: wanted 407, had 0
T1566: wanted 273, had 0
T1059: wanted 206, had 0
T1486: wanted 168, had 0
T1105: wanted 117, had 0
S0154: wanted 84, had 0
S0266: wanted 76, had 0
T1071: wanted 75, had 0
T1027: wanted 65, had 0
T1140: wanted 64, had 0
S0534: wanted 61, had 0
G0044: wanted 55, had 0
G0059: wanted 46, had 0
S0367: wanted 40, had 0
TA0006: wanted 39, had 0
G0007: wanted 39, had 0
TA0010: wanted 38, had 0
S0648: wanted 34, had 0
TA0003: wanted 30, had 0
TA0002: wanted 30, had 0
G0003: wanted 30, had 0
G0065: wanted 27, had 0
G0080: wanted 21, had 0
S0198: wanted 19, had 0
S0262: wanted 19, had 0
S0496: wanted 17, had 0
TA0009: wanted 17, had 0
TA0005: wanted 14, had 0
S0446: wanted 13, had 0
S0332: wanted 12, had 0
TA0001: wanted 12, had 0
S0650: wanted 12, had 0
G0096: wante

/tmp/ipykernel_2030815/1980070467.py:101: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  augmented_df['document'].fillna('artificial_data', inplace=True)


In [38]:
import json
import pandas as pd

OUT_JSON = "datasets/bosch_only_augmented_artificial_columnar.json"

def df_to_columnar_json(df: pd.DataFrame, out_path: str,
                        sentence_col="sentence", labels_col="labels",
                        doc_col_candidates=("doc_title", "document"), count=None):
    # pick a doc column name that exists, else create one
    doc_col = None
    for c in doc_col_candidates:
        if c in df.columns:
            doc_col = c
            break
    if doc_col is None:
        doc_col = doc_col_candidates[0]
        df[doc_col] = f"artificial_data_{count}" if count is not None else "artificial_data"

    # ensure no NaNs
    df = df.copy()
    df[sentence_col] = df[sentence_col].fillna("").astype(str)
    df[labels_col] = df[labels_col].apply(lambda x: x if isinstance(x, list) else [])
    df[doc_col] = df[doc_col].fillna("artificial_data").astype(str)

    out = {"sentence": {}, "labels": {}, "doc_title": {}}
    for i, row in df.reset_index(drop=True).iterrows():
        k = str(i)
        out["sentence"][k] = row[sentence_col]
        out["labels"][k] = row[labels_col]
        out["doc_title"][k] = row[doc_col]  # keep key name doc_title for compatibility

    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)

# ---- after you create artificial_df ----
# (you already did: artificial_df.drop_duplicates(subset=['sentence'], inplace=True))

# add/normalize doc title

    if "doc_title" in artificial_df.columns:
        artificial_df["doc_title"] = artificial_df["doc_title"].fillna(f"artificial_data_{count}")
    elif "document" in artificial_df.columns:
        artificial_df["document"] = artificial_df["document"].fillna(f"artificial_data_{count}")
    else:
        artificial_df["doc_title"] = f"artificial_data_{count}"
    count+=1
# shuffle (optional)
artificial_df_shuffled = artificial_df.sample(frac=1, random_state=random_state).reset_index(drop=True)

artificial_df_shuffled["doc_title"] = [
    f"artificial_data_{i}" for i in range(len(artificial_df_shuffled))
]
count=0
# write ONLY augmented set in columnar format
df_to_columnar_json(artificial_df_shuffled, OUT_JSON, count=count)

print("Wrote:", OUT_JSON, "rows:", len(artificial_df_shuffled))


Wrote: datasets/bosch_only_augmented_artificial_columnar.json rows: 928


Training val overlapping    

In [3]:
import hashlib
import pandas as pd

def text_hash(s):
    return hashlib.sha256(s.strip().encode("utf-8")).hexdigest()

# Load
df_original = pd.read_json("datasets/bosch_train.json")
df_val = pd.read_json("datasets/bosch_split_val_hmcat.json")

# Create hash helper column
df_original["_s"] = df_original["sentence"].apply(text_hash)
df_val["_s"] = df_val["sentence"].apply(text_hash)

# Remove sentence overlap
val_hashes = set(df_val["_s"])
df_train_clean = df_original[~df_original["_s"].isin(val_hashes)].copy()

# Drop helper column ONLY
df_train_clean = df_train_clean.drop(columns=["_s"])


In [4]:
print(df_train_clean.columns)
df_train_clean.to_json(
    "datasets/bosch_train_no_val_overlap_hmcat.json",
    orient="columns",
    indent=2
)



Index(['sentence', 'labels', 'entities', 'document'], dtype='object')
